Instalasi Library

In [15]:
# Install semua library dasar secara bersamaan (-q untuk quiet mode agar output tidak terlalu panjang)
!pip install -q -U \
    langchain \
    langchain-core \
    langchain-community \
    langchain-openai \
    langgraph \
    chromadb \
    pypdf \
    tiktoken \
    neo4j \
    openai \
    langchain-text-splitters

 Import & Integrasi Storage

In [16]:
# Import library standar dan LangChain
import re
from pypdf import PdfReader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document
from google.colab import drive

# Mount Google Drive (Tidak perlu files.upload lagi)
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


Data Loading & Ekstraksi

In [17]:
# Tentukan path PDF
pdf_path = "/content/drive/MyDrive/semester 6/data-skripsi/Terjemahan Tahafut Al-Falasifah (SFILE.MOBI).pdf"

# Baca PDF
reader = PdfReader(pdf_path)
print(f"Total halaman: {len(reader.pages)}")

# Ekstrak teks
pages = []
for i, page in enumerate(reader.pages):
    text = page.extract_text()

    # Validasi: pastikan teks tidak None dan tidak kosong
    if text and text.strip():
        pages.append({
            "page_number": i + 1,  # PDF pagination
            "text": text.strip()   # Hilangkan whitespace ekstra di awal/akhir
        })

# Cek hasil 1000 karakter pertama dari halaman pertama yang berhasil diekstrak
if pages:
    print("\n--- Cuplikan Teks ---")
    print(pages[0]["text"][:1000])
else:
    print("Tidak ada teks yang berhasil diekstrak.")

Total halaman: 454

--- Cuplikan Teks ---
pustaka-indo.blogspot.com


Alternatif LangChain

In [18]:
# Alternatif menggunakan PyPDFLoader (langsung menjadi format Document LangChain)
from langchain_community.document_loaders import PyPDFLoader

pdf_path = "/content/drive/MyDrive/semester 6/data-skripsi/Terjemahan Tahafut Al-Falasifah (SFILE.MOBI).pdf"
loader = PyPDFLoader(pdf_path)
documents = loader.load()

print(f"Total halaman: {len(documents)}")
print(documents[0].page_content[:1000])

Total halaman: 454
pustaka-indo.blogspot.com


preprocessing

In [19]:
import re
from pypdf import PdfReader
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter

# ==========================================
# 1. KAMUS PEMETAAN MASALAH TAHAFUT
# ==========================================
KATEGORI_TAHAFUT = {
    "MASALAH PERTAMA": {"topik": "Kosmologi (Keabadian Alam)", "judul": "Sanggahan Atas Pandangan Para Filsuf Tentang Eternitas Alam"},
    "MASALAH KEDUA": {"topik": "Kosmologi (Ruang & Waktu)", "judul": "Penolakan Terhadap Keyakinan Para Filsuf Atas Keabadian Alam, Ruang, dan Waktu"},
    "MASALAH KETIGA": {"topik": "Teologi (Penciptaan)", "judul": "Ketidakjujuran Para Filsuf Bahwa Tuhan Adalah Pencipta Alam Dan Penjelasan Bahwa Ungkapan Tersebut Hanya Bersifat Metaforis"},
    "MASALAH KEEMPAT": {"topik": "Teologi (Eksistensi Tuhan)", "judul": "Penjelasan Tentang Ketidakmampuan Filsuf Membuktikan Eksistensi Pencipta Alam"},
    "MASALAH KELIMA": {"topik": "Teologi (Keesaan Tuhan)", "judul": "Ketakmampuan Para Filsuf Membangun Argumen Keesaan Tuhan Dan Ketakmungkinan Penetapan Dua Wajib Al-Wujud Yang Tanpa Sebab"},
    "MASALAH KEENAM": {"topik": "Teologi (Sifat Tuhan)", "judul": "Sanggahan Atas Pandangan Para Filsuf Tentang Negasi Sifat-Sifat Tuhan"},
    "MASALAH KETUJUH": {"topik": "Teologi (Zat Tuhan)", "judul": "Bahwa Mustahil Tuhan Bersama Yang Lain Dalam Genus Dan Berpisah Dari Diferensia, Dan Dia Bukanlah Genus Atau Diferensia"},
    "MASALAH KEDELAPAN": {"topik": "Teologi (Kuiditas)", "judul": "Teori Bahwa Wujud-Nya Sederhana; Murni, Tanpa Kuiditas, dan Al-Wujud Al-Wajib Bagi-Nya Seperti Kuiditas Bagi Wujud Lainnya"},
    "MASALAH KESEMBILAN": {"topik": "Teologi (Ketubuhan)", "judul": "Tentang Ketidakmampuan Para Filsuf Untuk Membuktikan Melalui Argumen Rasional Bahwa Tuhan Bukan Tubuh (Jisim)"},
    "MASALAH KESEPULUH": {"topik": "Teologi (Penciptaan)", "judul": "Ketakmampuan Para Filsuf Membuktikan Secara Rasional Bahwa Alam Memiliki Pencipta Dan Sebab"},
    "MASALAH KESEBELAS": {"topik": "Teologi (Pengetahuan Tuhan)", "judul": "Perkataan Sebagian Filsuf Bahwa Tuhan Mengetahui Yang Lainnya Serta Pelbagai Spesies Dan Genus Secara Universal"},
    "MASALAH KEDUA BELAS": {"topik": "Teologi (Pengetahuan Tuhan)", "judul": "Ketidakmampuan Para Filsuf Untuk Membuktikan Bahwa Tuhan Juga Mengetahui Zat-Nya Sendiri"},
    "MASALAH KETIGA BELAS": {"topik": "Teologi (Pengetahuan Tuhan)", "judul": "Bahwa Tuhan Tidak Mengetahui Tiap Partikularia Yang Dapat Dibagi Sesuai Dengan Pembagian Waktu 'Telah', 'Sedang', Dan 'Akan'"},
    "MASALAH KEEMPAT BELAS": {"topik": "Kosmologi (Gerak Langit)", "judul": "Ketakmampuan Para Filsuf Untuk Membuktikan Bahwa Langit Adalah Makhluk Hidup Yang Mematuhi Tuhan Melalui Gerak Putarnya"},
    "MASALAH KELIMA BELAS": {"topik": "Kosmologi (Gerak Langit)", "judul": "Sanggahan Atas Apa Yang Para Filsuf Sebut Tujuan Yang Menggerakkan Langit"},
    "MASALAH KEENAM BELAS": {"topik": "Kosmologi (Pengetahuan Langit)", "judul": "Sanggahan Terhadap Tesis Para Filsuf Bahwa Jiwa-jiwa Langit Mengetahui Semua Partikularia Yang Memiliki Awal Temporal (Al-Juz'iyyat Al-Hadisah) Dalam Alam"},
    "MASALAH KETUJUH BELAS": {"topik": "Kausalitas (Hukum Alam)", "judul": "Sanggahan Atas Keyakinan Para Filsuf Terhadap Kemustahilan Independensi Sebab Dan Akibat"},
    "MASALAH KEDELAPAN BELAS": {"topik": "Metafisika Jiwa (Spiritualitas)", "judul": "Kerancuan Argumen Teoretis Bahwa Jiwa Manusia Adalah Substansi Yang Ada Dengan Sendirinya, Tidak Menempati Ruang Dan Tubuh"},
    "MASALAH KESEMBILAN BELAS": {"topik": "Metafisika Jiwa (Keabadian)", "judul": "Tesis Para Filsuf Bahwa Jiwa Manusia Abadi Dan Mustahil Tiada Setelah Bereksistensi, Tidak Bisa Dibayangkan Kehancurannya"},
    "MASALAH KEDUA PULUH": {"topik": "Eskatologi (Kebangkitan Fisik)", "judul": "Penolakan Para Filsuf Atas Kebangkitan Jasad, Kembalinya Jiwa Ke Jasad, Eksistensi Fisik Surga Dan Neraka, Dan Segala Yang Dijanjikan Allah"}
}

# ==========================================
# 2. FUNGSI CLEANING UTAMA
# ==========================================
def clean_and_normalize(text: str) -> str:
    if not text: return ""

    text = re.sub(r"^\s*[xlvi]+[xlvi]*\s*$", "", text, flags=re.MULTILINE | re.IGNORECASE)
    text = re.sub(r"\b(RY|VN)\b", "", text)
    text = re.sub(r"pustaka-indo\.blogspot\.com", "", text, flags=re.IGNORECASE)
    text = re.sub(r"--- PAGE \d+ ---", "", text)
    text = re.sub(r"^\s*\d+\s*$", "", text, flags=re.MULTILINE)
    text = re.sub(r"(Kerancuan Filsafat \(Tahafut Al-Falasifah\)|Imam Al-Gazali)", "", text)
    text = re.sub(r"\b([A-Z])\s+([a-z]{2,})\b", r"\1\2", text)
    text = re.sub(r"\b([A-Z])\n([a-z]{2,})\b", r"\1\2", text)
    text = re.sub(r"(\w+)\s*-\s*\n\s*(\w+)", r"\1\2", text)
    text = re.sub(r"(?<!ke-)(?<=[a-zA-Z])\d{1,2}(?=\s|$|\.|,)", "", text)


    # Regex FINAL Filsuf (Ditambah tangkapan untuk "Menurut para filsuf,")
    pattern_filsuf = r"(?:(?:Jika|Apabila|Kalau)\s+(?:mereka\s+berkata|dikatakan|Anda\s*(?:katakan|menyatakan)|orang\s+berkata)|Para\s+filsuf\s+(?:mengatakan|menyatakan|berpegang\s+teguh\s+dengan\s+pandangan)|Menurut\s+para\s+filsuf|Dia\s+mengatakan)(?:[:\s,]|\s+bahwa)"
    text = re.sub(pattern_filsuf, r"\n\n[ARGUMEN FILSUF]: ", text, flags=re.IGNORECASE)


    # Regex FINAL Al-Ghazali (Update tambahan untuk kata kerja pasif "disanggah")
    pattern_ghazali = r"(?:Kami\s+menjawab|Jawaban(?:nya)?|Sanggahan(?:nya)?[^:\.]*|Kami\s+akan\s+menjawab|Akan\s+kami\s+katakan|Pembelaan\s+ini\s+menjadi\s+absurd[^:\.]*|Inilah\s+cara\s+mengatasi\s+argumen(?:-argumen)?\s+mereka|Argumen\s+ini\s+dapat\s+disanggah[^:\.]*)[:\.\s]"
    text = re.sub(pattern_ghazali, r"\n\n[BANTAHAN GHAZALI]: ", text, flags=re.IGNORECASE)
    # ====================================================
    # TAMBAHKAN BLOK KAMUS ANTI-TYPO INI DI SINI
    # ====================================================
    # 1. AUTOKOREKSI KATA ULANG (Otomatis tangkap pola ABCDABCD -> ABCD-ABCD)
    text = re.sub(r"\b([a-z]{3,})\1\b", r"\1-\1", text, flags=re.IGNORECASE)

    # 2. KAMUS TYPO KRITIS (Hanya untuk pemicu argumen)
    typo_dictionary = {
        "Andakatakan": "Anda katakan",
        "merekaberkata": "mereka berkata",
        "parafilsuf": "para filsuf",
        "Diamengatakan": "Dia mengatakan"
    }
    for salah, benar in typo_dictionary.items():
        text = re.sub(salah, benar, text, flags=re.IGNORECASE)
    # ====================================================

    # Mengubah tanda pisah panjang (em-dash) menjadi spasi agar kalimat mengalir natural
    text = text.replace("—", " ")
    text = re.sub(r'\n\s*\n', '\n\n', text)
    text = re.sub(r'[ \t]+', ' ', text)
    text = re.sub(r"(MASALAH\s+\w+(?:\s+\w+)?)", r"\n\n\1\n", text, flags=re.IGNORECASE)
    text = re.sub(r"\[ARGUMEN FILSUF\]", "\n\n[ARGUMEN FILSUF]\n", text)
    text = re.sub(r"\[BANTAHAN GHAZALI\]", "\n\n[BANTAHAN GHAZALI]\n", text)

    return text.strip()

# ==========================================
# 3. EKSEKUSI BACA PDF & CLEANING
# ==========================================
pdf_path = "/content/drive/MyDrive/semester 6/data-skripsi/Terjemahan Tahafut Al-Falasifah (SFILE.MOBI).pdf"
cleaned_pages = []

try:
    reader = PdfReader(pdf_path)
    print(f"Total halaman PDF ditemukan: {len(reader.pages)}")

    for i, page in enumerate(reader.pages):
        pg_num = i + 1
        if (63 <= pg_num <= 78) or (pg_num >= 83):
            raw_text = page.extract_text()
            if raw_text:
                clean_text = clean_and_normalize(raw_text)
                if len(clean_text) > 50:
                    cleaned_pages.append({
                        "page_number": pg_num,
                        "text": clean_text,
                        "source": "Tahafut Al-Falasifah"
                    })

    print(f"✅ Berhasil mengekstrak {len(cleaned_pages)} halaman bersih.")

    # ========================================================
    # BAGIAN YANG TERTINGGAL: SIMPAN KE TXT UNTUK PENGECEKAN
    # ========================================================
    final_text = "\n\n".join([p["text"] for p in cleaned_pages])
    output_filename = "Tahafut_Al_Falasifah_CLEAN_FINAL.txt"
    with open(output_filename, "w", encoding="utf-8") as f:
        f.write(final_text)
    print(f"📥 File hasil cleaning disimpan ke: {output_filename}")
    # ========================================================


except FileNotFoundError:
    print(f"❌ Error: File PDF tidak ditemukan di path:\n{pdf_path}")
    print("Pastikan Google Drive sudah di-mount dan path-nya benar.")
except Exception as e:
    print(f"❌ Terjadi kesalahan saat membaca PDF: {e}")

# ==========================================
# 4. PEMBUATAN METADATA & DOCUMENT LANGCHAIN
# ==========================================
if cleaned_pages:
    documents = []

    # State awal metadata
    current_masalah = "Mukadimah"
    current_topik = "Pengantar Filsafat"
    current_judul = "Pengantar"
    current_label = "Pembahasan Umum"
    current_tokoh = "Al-Ghazali"

    for page_data in cleaned_pages:
        page_text = page_data["text"]
        page_number = page_data["page_number"]
        source = page_data.get("source", "Tahafut Al-Falasifah")

        text_upper = page_text.upper()

        # Deteksi Masalah menggunakan Regex yang lebih kuat (menangkap 1 atau 2 kata)
        match = re.search(r"MASALAH\s+([A-Z]+\s+PULUH|[A-Z]+\s+BELAS|[A-Z]+)", text_upper)
        if match:
            # Membersihkan spasi ganda hasil tangkapan regex jika ada
            kunci_masalah = " ".join(match.group().split())

            if kunci_masalah in KATEGORI_TAHAFUT:
                current_masalah = kunci_masalah
                current_topik = KATEGORI_TAHAFUT[kunci_masalah]["topik"]
                current_judul = KATEGORI_TAHAFUT[kunci_masalah]["judul"]

        # Deteksi Label & Tokoh
        if "[BANTAHAN GHAZALI]" in text_upper:
            current_label = "Bantahan"
            current_tokoh = "Al-Ghazali"
        elif "[ARGUMEN FILSUF]" in text_upper:
            current_label = "Argumen"
            current_tokoh = "Filsuf"

        documents.append(
            Document(
                page_content=page_text,
                metadata={
                    "page": page_number,
                    "masalah": current_masalah,
                    "judul_masalah": current_judul,
                    "label": current_label,
                    "tokoh": current_tokoh,
                    "topik": current_topik,
                    "sumber": source
                }
            )
        )

    print(f"✅ Total Document LangChain dibuat: {len(documents)}")

    # ==========================================
    # 5. CHUNKING TEKS
    # ==========================================
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=1100,
        chunk_overlap=200,
        separators=[
            "\n[BANTAHAN GHAZALI]",
            "\n[ARGUMEN FILSUF]",
            "\nMASALAH",
            "\n\n",
            "\n",
            " ",
            ""
        ]
    )

    docs_split = splitter.split_documents(documents)

    # Tambahkan ID pada setiap potongan
    for idx, doc in enumerate(docs_split):
        doc.metadata["chunk_id"] = idx

    print(f"✅ Total chunk yang siap diproses: {len(docs_split)}")

    # Preview satu chunk untuk memastikan metadata masuk dengan benar
    if docs_split:
        print("\n--- PREVIEW METADATA CHUNK PERTAMA ---")
        for key, value in docs_split[0].metadata.items():
            print(f"{key}: {value}")
        print("\n--- TEKS ---")
        print(f"{docs_split[0].page_content[:300]}...\n")
else:
    print("⚠️ Proses Document dibatalkan karena cleaned_pages kosong.")

Total halaman PDF ditemukan: 454
✅ Berhasil mengekstrak 376 halaman bersih.
📥 File hasil cleaning disimpan ke: Tahafut_Al_Falasifah_CLEAN_FINAL.txt
✅ Total Document LangChain dibuat: 376
✅ Total chunk yang siap diproses: 813

--- PREVIEW METADATA CHUNK PERTAMA ---
page: 63
masalah: Mukadimah
judul_masalah: Pengantar
label: Pembahasan Umum
tokoh: Al-Ghazali
topik: Pengantar Filsafat
sumber: Tahafut Al-Falasifah
chunk_id: 0

--- TEKS ---
Mukadimah

Kami memohon pada Allah dengan keagungan-Nya 
yang melebihi segala batas akhir dan keperkasaan-Nya 
yang melintasi segala ukuran agar melimpahkan cahaya 
petunjuk kepada kami dan menyelamatkan kami dari gulita 
kesesatan, agar menjadikan kami termasuk orang yang melihat 
kebenaran sebagai...



In [ ]:
import os
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import Chroma

# Pastikan API key sudah di set
os.environ["OPENAI_API_KEY"] = ""

# Buat embeddings


embeddings = OpenAIEmbeddings(
    model="text-embedding-3-large"
)

vectordb = Chroma.from_documents(
    documents=docs_split,
    embedding=embeddings,
    persist_directory="chroma_db_tahafut"
)

vectordb.persist()

print("✅ Data berhasil masuk ke Chroma")



✅ Data berhasil masuk ke Chroma


In [21]:
query = "Kerancuan pemikiran filsafat menurut Al-Ghazali"

docs = vectordb.max_marginal_relevance_search(
    query,
    k=3,
    fetch_k=10
)

print("\n=== Context dari Chroma ===\n")

for i, d in enumerate(docs):
    print(f"[{i+1}]")
    print("Masalah       :", d.metadata.get("masalah", "-"))
    print("Judul Masalah :", d.metadata.get("judul_masalah", "-")) # <-- Tambahan
    print("Topik         :", d.metadata.get("topik", "-"))         # <-- Tambahan
    print("Label         :", d.metadata.get("label", "-"))
    print("Tokoh         :", d.metadata.get("tokoh", "-"))
    print("Sumber        :", d.metadata.get("sumber", "-"))        # <-- Tambahan
    print("Page          :", d.metadata.get("page", "-"))
    print("Chunk ID      :", d.metadata.get("chunk_id", "-"))      # <-- Tambahan
    print("Teks          :\n", d.page_content[:300], "...\n")
    print("-" * 50)


=== Context dari Chroma ===

[1]
Masalah       : MASALAH KEENAM
Judul Masalah : Sanggahan Atas Pandangan Para Filsuf Tentang Negasi Sifat-Sifat Tuhan
Topik         : Teologi (Sifat Tuhan)
Label         : Bantahan
Tokoh         : Al-Ghazali
Sumber        : Tahafut Al-Falasifah
Page          : 247
Chunk ID      : 396
Teks          :
 tentang objek ini. Sebab apabila dia ingat pada pengetahuannya, 
ia akan menuntut pengetahuan lain, yang dengannya ingatannya 
akan lenyap.
Mengenai pernyataan mereka bahwa 



[BANTAHAN GHAZALI]
: Karena alasan inilah, kami menamakan buku ini 
Tahafut al-Falasifah (Kerancuan Para Filsuf), bukan Tam ...

--------------------------------------------------
[2]
Masalah       : MASALAH KELIMA
Judul Masalah : Ketakmampuan Para Filsuf Membangun Argumen Keesaan Tuhan Dan Ketakmungkinan Penetapan Dua Wajib Al-Wujud Yang Tanpa Sebab
Topik         : Teologi (Keesaan Tuhan)
Label         : Bantahan
Tokoh         : Al-Ghazali
Sumber        : Tahafut Al-Falasifah
Page  

In [22]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

def rag_chat(query, k=3):
    # =========================
    # RETRIEVAL CHROMA (MMR)
    # =========================
    retrieved_docs = vectordb.max_marginal_relevance_search(
        query,
        k=k,
        fetch_k=10
    )

    # =========================
    # BUILD CONTEXT + METADATA LENGKAP
    # =========================
    context_parts = []

    for i, d in enumerate(retrieved_docs):
        # INI DIA SUSUNAN METADATA LENGKAP ANDA:
        meta = (
            f"[{i+1}]\n"
            f"Masalah       : {d.metadata.get('masalah', '-')}\n"
            f"Judul Masalah : {d.metadata.get('judul_masalah', '-')}\n"
            f"Topik         : {d.metadata.get('topik', '-')}\n"
            f"Label         : {d.metadata.get('label', '-')}\n"
            f"Tokoh         : {d.metadata.get('tokoh', '-')}\n"
            f"Sumber        : {d.metadata.get('sumber', '-')}\n"
            f"Page          : {d.metadata.get('page', '-')}\n"
            f"Chunk ID      : {d.metadata.get('chunk_id', '-')}\n"
        )

        context_parts.append(meta + f"Teks          :\n{d.page_content}")

    context = "\n\n".join(context_parts)

    # =========================
    # PROMPT AKADEMIK
    # =========================
    prompt = f"""
    Anda adalah asisten akademik filsafat Islam.

    Gunakan konteks berikut untuk menjawab pertanyaan secara ilmiah,
    sistematis, dan berbasis teks Tahafut al-Falasifah.

    Struktur jawaban WAJIB sebagai berikut:
    1. Penjelasan Konsep
    2. Argumen Filsuf
    3. Bantahan Al-Ghazali
    4. Referensi (Halaman PDF)

    Jika salah satu bagian tidak ditemukan dalam konteks,
    tuliskan "Tidak ditemukan dalam Tahafut al-Falasifah".

    Konteks:
    {context}

    Pertanyaan:
    {query}

    Jawaban:
    """

    print("=== DEBUG CONTEXT ===")
    print(context[:1500]) # Saya panjangkan sedikit agar kelihatan semua metadatanya
    print("\n=====================\n")

    response = llm.invoke(prompt)
    return response.content

# ==========================================
# PENGUJIAN FUNGSI RAG
# ==========================================
pertanyaan = "Bagaimana argumen filsuf tentang penyusutan matahari, dan bagaimana Al-Ghazali membantahnya?"

print(f"Anda bertanya: {pertanyaan}\n")
print("Sedang mencari di dokumen dan menyusun jawaban AI...\n")

jawaban_final = rag_chat(pertanyaan, k=4)

print("\n" + "="*50)
print("🤖 HASIL JAWABAN AI:")
print("="*50 + "\n")
print(jawaban_final)

Anda bertanya: Bagaimana argumen filsuf tentang penyusutan matahari, dan bagaimana Al-Ghazali membantahnya?

Sedang mencari di dokumen dan menyusun jawaban AI...

=== DEBUG CONTEXT ===
[1]
Masalah       : MASALAH KEDUA
Judul Masalah : Penolakan Terhadap Keyakinan Para Filsuf Atas Keabadian Alam, Ruang, dan Waktu
Topik         : Kosmologi (Ruang & Waktu)
Label         : Bantahan
Tokoh         : Al-Ghazali
Sumber        : Tahafut Al-Falasifah
Page          : 145
Chunk ID      : 175
Teks          :
[BANTAHAN GHAZALI]
: terhadap semuanya ini telah dikemukakan di 
depan. Namun, kami letakkan 

masalah ini di
 tempat tersendiri, 
sebab mereka juga mempunyai dua buah argumen yang lain.
Pertama:
Argumen pertama adalah yang dipegang oleh Galen. 



[ARGUMEN FILSUF]
: apabila matahari dapat lenyap (rusak), tanda-tanda 
kerusakannya mesti melalui penyusutan. Tetapi pengamatan 
astronomis mengenai ukurannya selama beribu-ribu tahun 
hanya menunjukkan kuantitas seperti adanya (tanpa ada gejala 
pen

In [23]:
!pip install neo4j


In [ ]:
import os

NEO4J_URI = ""     # kalau Neo4j lokal
NEO4J_USER = "neo4j"
NEO4J_PASSWORD = ""

os.environ["NEO4J_URI"] = NEO4J_URI
os.environ["NEO4J_USER"] = NEO4J_USER
os.environ["NEO4J_PASSWORD"] = NEO4J_PASSWORD


In [25]:
from neo4j import GraphDatabase

driver = GraphDatabase.driver(
    os.environ["NEO4J_URI"],
    auth=(os.environ["NEO4J_USER"], os.environ["NEO4J_PASSWORD"])
)

def test_connection():
    with driver.session() as session:
        greeting = session.run("RETURN 'Neo4j Connected!' as msg").single()
        print(greeting["msg"])

test_connection()


Neo4j Connected!


In [26]:
from neo4j import GraphDatabase

# 1. Fungsi Utama untuk Membuat Struktur Graph
def create_tahafut_graph(tx):
    # Data Kamus Lengkap (Koleksi Metadata Anda)
    kategori =  {
    "MASALAH PERTAMA": {"topik": "Kosmologi (Keabadian Alam)", "judul": "Sanggahan Atas Pandangan Para Filsuf Tentang Eternitas Alam"},
    "MASALAH KEDUA": {"topik": "Kosmologi (Ruang & Waktu)", "judul": "Penolakan Terhadap Keyakinan Para Filsuf Atas Keabadian Alam, Ruang, dan Waktu"},
    "MASALAH KETIGA": {"topik": "Teologi (Penciptaan)", "judul": "Ketidakjujuran Para Filsuf Bahwa Tuhan Adalah Pencipta Alam Dan Penjelasan Bahwa Ungkapan Tersebut Hanya Bersifat Metaforis"},
    "MASALAH KEEMPAT": {"topik": "Teologi (Eksistensi Tuhan)", "judul": "Penjelasan Tentang Ketidakmampuan Filsuf Membuktikan Eksistensi Pencipta Alam"},
    "MASALAH KELIMA": {"topik": "Teologi (Keesaan Tuhan)", "judul": "Ketakmampuan Para Filsuf Membangun Argumen Keesaan Tuhan Dan Ketakmungkinan Penetapan Dua Wajib Al-Wujud Yang Tanpa Sebab"},
    "MASALAH KEENAM": {"topik": "Teologi (Sifat Tuhan)", "judul": "Sanggahan Atas Pandangan Para Filsuf Tentang Negasi Sifat-Sifat Tuhan"},
    "MASALAH KETUJUH": {"topik": "Teologi (Zat Tuhan)", "judul": "Bahwa Mustahil Tuhan Bersama Yang Lain Dalam Genus Dan Berpisah Dari Diferensia, Dan Dia Bukanlah Genus Atau Diferensia"},
    "MASALAH KEDELAPAN": {"topik": "Teologi (Kuiditas)", "judul": "Teori Bahwa Wujud-Nya Sederhana; Murni, Tanpa Kuiditas, dan Al-Wujud Al-Wajib Bagi-Nya Seperti Kuiditas Bagi Wujud Lainnya"},
    "MASALAH KESEMBILAN": {"topik": "Teologi (Ketubuhan)", "judul": "Tentang Ketidakmampuan Para Filsuf Untuk Membuktikan Melalui Argumen Rasional Bahwa Tuhan Bukan Tubuh (Jisim)"},
    "MASALAH KESEPULUH": {"topik": "Teologi (Penciptaan)", "judul": "Ketakmampuan Para Filsuf Membuktikan Secara Rasional Bahwa Alam Memiliki Pencipta Dan Sebab"},
    "MASALAH KESEBELAS": {"topik": "Teologi (Pengetahuan Tuhan)", "judul": "Perkataan Sebagian Filsuf Bahwa Tuhan Mengetahui Yang Lainnya Serta Pelbagai Spesies Dan Genus Secara Universal"},
    "MASALAH KEDUA BELAS": {"topik": "Teologi (Pengetahuan Tuhan)", "judul": "Ketidakmampuan Para Filsuf Untuk Membuktikan Bahwa Tuhan Juga Mengetahui Zat-Nya Sendiri"},
    "MASALAH KETIGA BELAS": {"topik": "Teologi (Pengetahuan Tuhan)", "judul": "Bahwa Tuhan Tidak Mengetahui Tiap Partikularia Yang Dapat Dibagi Sesuai Dengan Pembagian Waktu 'Telah', 'Sedang', Dan 'Akan'"},
    "MASALAH KEEMPAT BELAS": {"topik": "Kosmologi (Gerak Langit)", "judul": "Ketakmampuan Para Filsuf Untuk Membuktikan Bahwa Langit Adalah Makhluk Hidup Yang Mematuhi Tuhan Melalui Gerak Putarnya"},
    "MASALAH KELIMA BELAS": {"topik": "Kosmologi (Gerak Langit)", "judul": "Sanggahan Atas Apa Yang Para Filsuf Sebut Tujuan Yang Menggerakkan Langit"},
    "MASALAH KEENAM BELAS": {"topik": "Kosmologi (Pengetahuan Langit)", "judul": "Sanggahan Terhadap Tesis Para Filsuf Bahwa Jiwa-jiwa Langit Mengetahui Semua Partikularia Yang Memiliki Awal Temporal (Al-Juz'iyyat Al-Hadisah) Dalam Alam"},
    "MASALAH KETUJUH BELAS": {"topik": "Kausalitas (Hukum Alam)", "judul": "Sanggahan Atas Keyakinan Para Filsuf Terhadap Kemustahilan Independensi Sebab Dan Akibat"},
    "MASALAH KEDELAPAN BELAS": {"topik": "Metafisika Jiwa (Spiritualitas)", "judul": "Kerancuan Argumen Teoretis Bahwa Jiwa Manusia Adalah Substansi Yang Ada Dengan Sendirinya, Tidak Menempati Ruang Dan Tubuh"},
    "MASALAH KESEMBILAN BELAS": {"topik": "Metafisika Jiwa (Keabadian)", "judul": "Tesis Para Filsuf Bahwa Jiwa Manusia Abadi Dan Mustahil Tiada Setelah Bereksistensi, Tidak Bisa Dibayangkan Kehancurannya"},
    "MASALAH KEDUA PULUH": {"topik": "Eskatologi (Kebangkitan Fisik)", "judul": "Penolakan Para Filsuf Atas Kebangkitan Jasad, Kembalinya Jiwa Ke Jasad, Eksistensi Fisik Surga Dan Neraka, Dan Segala Yang Dijanjikan Allah"}
}

    # Query Cypher: Membuat Node Tokoh, Masalah, dan Topik serta Relasinya
    for kunci, data in kategori.items():
        tx.run("""
            MERGE (g:Tokoh {nama: 'Al-Ghazali'})
            MERGE (f:Tokoh {nama: 'Para Filsuf'})
            MERGE (m:Masalah {id: $id})
            ON CREATE SET m.judul = $judul
            MERGE (t:Topik {nama: $topik})

            MERGE (g)-[:MENULIS]->(m)
            MERGE (g)-[:MENYANGGAH]->(f)
            MERGE (f)-[:BERPENDAPAT_DI]->(m)
            MERGE (m)-[:BAGIAN_DARI_TOPIK]->(t)
        """, id=kunci, judul=data['judul'], topik=data['topik'])

# 2. Menjalankan Fungsi menggunakan Driver yang sudah Anda buat sebelumnya
try:
    with driver.session() as session:
        session.execute_write(create_tahafut_graph)
        print("✅ Berhasil! Knowledge Graph Tahafut Al-Falasifah telah dibangun di Neo4j.")
except Exception as e:
    print(f"❌ Terjadi kesalahan saat menulis ke Neo4j: {e}")

✅ Berhasil! Knowledge Graph Tahafut Al-Falasifah telah dibangun di Neo4j.


In [27]:
def upload_chunks_to_neo4j_hybrid(tx, chunks):
    for i, doc in enumerate(chunks):
        m = doc.metadata

        tx.run("""
            MATCH (m:Masalah {id: $masalah_id})
            MERGE (c:Chunk {chunk_id: $chunk_id})
            ON CREATE SET
                c.text = $content,
                c.page = $page_num,
                c.label = $label,
                c.tokoh = $tokoh,
                c.sumber = $sumber
            MERGE (c)-[:BAGIAN_DARI_TEKS]->(m)
        """,
        masalah_id=m.get("masalah", "Mukadimah"),
        chunk_id=m.get("chunk_id", i),
        content=doc.page_content,
        page_num=m.get("page", 0),
        label=m.get("label", "-"),
        tokoh=m.get("tokoh", "-"),
        sumber=m.get("sumber", "Tahafut Al-Falasifah"))

# Jalankan ini untuk memasukkan TEKS detail ke dalam GRAF
try:
    with driver.session() as session:
        print(f"Menghubungkan {len(docs_split)} potongan teks ke dalam struktur graf...")
        session.execute_write(upload_chunks_to_neo4j_hybrid, docs_split)
        print("✅ SELESAI! Sekarang Graf Anda punya 'Struktur' DAN 'Isi Teks'.")
except Exception as e:
    print(f"❌ Error: {e}")

Menghubungkan 813 potongan teks ke dalam struktur graf...
✅ SELESAI! Sekarang Graf Anda punya 'Struktur' DAN 'Isi Teks'.


In [28]:
def hybrid_rag_chat(query, k=5):
    # 1. RETRIEVAL DARI CHROMADB (VECTOR)
    vektor_docs = vectordb.max_marginal_relevance_search(query, k=k, fetch_k=10)

    # 2. RETRIEVAL DARI NEO4J (GRAPH)
    # Kita mencari berdasarkan kata kunci yang ada di query
    cypher_query = """
    MATCH (t:Topik)-[:BAGIAN_DARI_TOPIK]-(m:Masalah)-[:BAGIAN_DARI_TEKS]-(c:Chunk)
    WHERE toLower(t.nama) CONTAINS toLower($search)
       OR toLower(m.id) CONTAINS toLower($search)
       OR toLower(m.judul) CONTAINS toLower($search)
    RETURN
        m.id AS masalah_id,
        m.judul AS judul,
        t.nama AS topik,
        c.text AS teks,
        c.page AS halaman,
        c.label AS label,
        c.tokoh AS tokoh
    ORDER BY m.id ASC, c.page ASC
    LIMIT 10
    """

    graph_docs = []
    with driver.session() as session:
        # Kita masukkan query utuh ke Neo4j untuk mencari kecocokan topik/masalah
        result = session.run(cypher_query, search=query)
        graph_docs = [dict(record) for record in result]

    # 3. BUILD CONTEXT (MENGGABUNGKAN KEDUANYA)
    context_parts = []

    # Masukkan data dari Vector
    for i, d in enumerate(vektor_docs):
        meta = (
            f"[KONTEKS VEKTOR {i+1}]\n"
            f"Masalah       : {d.metadata.get('masalah', '-')}\n"
            f"Judul Masalah : {d.metadata.get('judul_masalah', '-')}\n"
            f"Topik         : {d.metadata.get('topik', '-')}\n"
            f"Label         : {d.metadata.get('label', '-')}\n"
            f"Tokoh         : {d.metadata.get('tokoh', '-')}\n"
            f"Halaman       : {d.metadata.get('page', '-')}\n"
        )
        context_parts.append(meta + f"Teks          : {d.page_content}")

    # Masukkan data dari Graph
    for i, g in enumerate(graph_docs):
        meta = (
            f"[KONTEKS GRAPH {i+1}]\n"
            f"ID Masalah    : {g['masalah_id']}\n"
            f"Judul Masalah : {g['judul']}\n"
            f"Topik         : {g['topik']}\n"
            f"Label         : {g['label']}\n"
            f"Halaman       : {g['halaman']}\n"
        )
        context_parts.append(meta + f"Teks          : {g['teks']}")

    context = "\n\n".join(context_parts)

    # 4. PROMPT AKADEMIK HYBRID (VERSI TAJAM)
    prompt = f"""
    Anda adalah asisten akademik filsafat Islam yang pakar dalam kitab "Tahafut al-Falasifah" karya Imam Al-Ghazali.

    Tugas Anda adalah menjawab pertanyaan berdasarkan dua sumber konteks:
    1. KONTEKS VEKTOR: Berisi potongan teks detail (mikro).
    2. KONTEKS GRAPH: Berisi struktur masalah dan daftar topik terkait (makro).

    STRUKTUR JAWABAN (WAJIB):
    1. Penjelasan Konsep: Terangkan definisi atau latar belakang masalah.
    2. Argumen Filsuf: Jelaskan posisi filsuf berdasarkan teks (misalnya: Ibnu Sina, Al-Farabi, Aristoteles, dll, atau cukup gunakan sebutan umum "para filsuf" jika teks tidak menyebut nama secara spesifik).
    3. Bantahan Al-Ghazali: Jelaskan secara sistematis bagaimana Al-Ghazali meruntuhkan argumen tersebut.
    4. Referensi Akademik: WAJIB sebutkan "ID Masalah" (Contoh: MASALAH PERTAMA) dan "Halaman PDF" untuk setiap poin penting.

    ATURAN KETAT:
    - Jika jawaban tidak ditemukan dalam konteks, katakan "Informasi tidak ditemukan dalam database Tahafut al-Falasifah".
    - Jangan mengarang informasi di luar konteks yang diberikan.
    - Prioritaskan ID Masalah yang berasal dari KONTEKS GRAPH untuk menyusun daftar.

    Konteks:
    {context}

    Pertanyaan:
    {query}

    Jawaban:
    """

    print("=== DEBUG HYBRID CONTEXT (Vektor + Graf) ===")
    print(context[:1500], "...")
    print("\n============================================\n")

    response = llm.invoke(prompt)
    return response.content

# ==========================================
# PENGUJIAN HYBRID RAG
# ==========================================
pertanyaan = "jelaskan mengenai kausalitas"
jawaban_hybrid = hybrid_rag_chat(pertanyaan)

print("\n🤖 HASIL JAWABAN HYBRID RAG:")
print(jawaban_hybrid)

=== DEBUG HYBRID CONTEXT (Vektor + Graf) ===
[KONTEKS VEKTOR 1]
Masalah       : MASALAH KETIGA
Judul Masalah : Ketidakjujuran Para Filsuf Bahwa Tuhan Adalah Pencipta Alam Dan Penjelasan Bahwa Ungkapan Tersebut Hanya Bersifat Metaforis
Topik         : Teologi (Penciptaan)
Label         : Argumen
Tokoh         : Filsuf
Halaman       : 173
Teks          : sesuatu yang tidak timbul dari ketiadaan hanya disebut perbuatan 
dalam pengertian metaforis dan sama sekali tak memiliki makna 
hakiki. Mengenai hubungan antara sebab dan akibat, keduanya 
mungkin bersifat kekal (qadim), atau keduanya bersifat temporal 
(hadis). Misalnya, dikatakan bahwa pengetahuan yang kekal 
adalah akibat dari adanya Tuhan yang Qadim (Kekal) sebagai 
yang mengetahui. Hal ini tak diperdebatkan. Yang diperdebatkan 
hanya berkisar tentang apa yang disebut perbuatan. Akibat dari 
suatu sebab tak disebut perbuatan dari sebab itu kecuali dalam 
pengertian metaforisnya. Karena sudah merupakan syarat bagi 
suatu perbuatan un

In [31]:
import numpy as np
import pandas as pd
from sklearn.metrics.pairwise import cosine_similarity

# ==========================================
# 1. FUNGSI EVALUASI (SESUAI REQUEST ANDA)
# ==========================================
def calculate_metrics(query, answer, context):
    if not answer or not context:
        return 0.0, 0.0, 0.0
    try:
        # 1. Embed Query dan Answer
        v_q = np.array(embeddings.embed_query(query)).reshape(1, -1)
        v_a = np.array(embeddings.embed_query(answer)).reshape(1, -1)

        relevance = cosine_similarity(v_q, v_a)[0][0]

        # 2. Pecah Konteks Menjadi Blok
        context_blocks = [c.strip() for c in context.split("\n\n") if len(c.strip()) > 50]
        if not context_blocks:
            context_blocks = [context]

        v_c_list = np.array(embeddings.embed_documents(context_blocks))

        # Faithfulness
        faithfulness_scores = cosine_similarity(v_a, v_c_list)
        faithfulness = np.max(faithfulness_scores)

        # Context Precision
        precision_scores = cosine_similarity(v_q, v_c_list)
        precision = np.mean(precision_scores)

        if "tidak ditemukan" in answer.lower() or "maaf" in answer.lower():
            faithfulness = 0.0
            relevance = 0.0

        return round(float(faithfulness), 3), round(float(relevance), 3), round(float(precision), 3)
    except Exception as e:
        print(f"Error perhitungan metrik: {e}")
        return 0.0, 0.0, 0.0

# ==========================================
# 2. LOOP EVALUASI (TANPA UBAH FUNGSI ASLI)
# ==========================================

# Daftar pertanyaan untuk uji skripsi
daftar_uji = [
    "jelaskan mengenai kausalitas",
    "apa argumen filsuf mengenai kebaqaan jiwa",
    "apakah alam itu qadim atau baru"
]

hasil_skripsi = []

print("=== MEMULAI EVALUASI METRIKS SKRIPSI ===")

for q in daftar_uji:
    print(f"\nProses Pertanyaan: {q}")

    # --- A. Ambil Jawaban dari Fungsi Asli Anda ---
    jawaban = hybrid_rag_chat(q)

    # --- B. Ambil Konteks Secara Manual (Sama dengan logika di fungsi Anda) ---
    # Ini supaya kita punya variabel 'context' untuk dihitung metriknya
    v_docs = vectordb.max_marginal_relevance_search(q, k=5, fetch_k=10)

    cypher_eval = """
    MATCH (t:Topik)-[:BAGIAN_DARI_TOPIK]-(m:Masalah)-[:BAGIAN_DARI_TEKS]-(c:Chunk)
    WHERE toLower(t.nama) CONTAINS toLower($search)
       OR toLower(m.id) CONTAINS toLower($search)
       OR toLower(m.judul) CONTAINS toLower($search)
    RETURN c.text AS teks LIMIT 10
    """
    with driver.session() as session:
        result = session.run(cypher_eval, search=q)
        g_teks = [record["teks"] for record in result]

    # Gabungkan semua teks konteks untuk evaluasi
    konteks_untuk_eval = "\n\n".join([d.page_content for d in v_docs] + g_teks)

    # --- C. Hitung Metrik ---
    f, r, p = calculate_metrics(q, jawaban, konteks_untuk_eval)

    hasil_skripsi.append({
        "Pertanyaan": q,
        "Faithfulness": f,
        "Relevance": r,
        "Precision": p
    })

# ==========================================
# 3. OUTPUT HASIL
# ==========================================
df_hasil = pd.DataFrame(hasil_skripsi)

print("\n" + "="*50)
print("HASIL EVALUASI UNTUK BAB 4 SKRIPSI")
print("="*50)
print(df_hasil)

print("\nSKOR RATA-RATA:")
print(df_hasil.mean(numeric_only=True))

=== MEMULAI EVALUASI METRIKS SKRIPSI ===

Proses Pertanyaan: jelaskan mengenai kausalitas
=== DEBUG HYBRID CONTEXT (Vektor + Graf) ===
[KONTEKS VEKTOR 1]
Masalah       : MASALAH KETIGA
Judul Masalah : Ketidakjujuran Para Filsuf Bahwa Tuhan Adalah Pencipta Alam Dan Penjelasan Bahwa Ungkapan Tersebut Hanya Bersifat Metaforis
Topik         : Teologi (Penciptaan)
Label         : Argumen
Tokoh         : Filsuf
Halaman       : 173
Teks          : sesuatu yang tidak timbul dari ketiadaan hanya disebut perbuatan 
dalam pengertian metaforis dan sama sekali tak memiliki makna 
hakiki. Mengenai hubungan antara sebab dan akibat, keduanya 
mungkin bersifat kekal (qadim), atau keduanya bersifat temporal 
(hadis). Misalnya, dikatakan bahwa pengetahuan yang kekal 
adalah akibat dari adanya Tuhan yang Qadim (Kekal) sebagai 
yang mengetahui. Hal ini tak diperdebatkan. Yang diperdebatkan 
hanya berkisar tentang apa yang disebut perbuatan. Akibat dari 
suatu sebab tak disebut perbuatan dari sebab itu kecu

In [32]:
# import pandas as pd
# import json

# def evaluasi_ragas_judge(query, ans_biasa, ans_hybrid, context_biasa, context_hybrid):
#     """
#     Fungsi Evaluator untuk membandingkan RAG Biasa vs Hybrid RAG
#     berdasarkan 3 Metrik Utama RAGAS.
#     """
#     prompt_eval = f"""
#     Anda adalah Pakar Evaluator Sistem RAG. Berikan skor numerik antara 0.0 hingga 1.0
#     untuk dua sistem (A dan B) berdasarkan kriteria RAGAS berikut:

#     1. Faithfulness: Seberapa setia jawaban terhadap konteks yang diberikan (tidak halusinasi).
#     2. Answer Relevance: Seberapa relevan dan lengkap jawaban terhadap pertanyaan.
#     3. Context Precision: Seberapa tepat konteks yang ditarik mendukung jawaban yang benar.

#     PERTANYAAN: {query}

#     ---------------------------------------------------------------------------
#     [SISTEM A - RAG BIASA]
#     Jawaban: {ans_biasa}
#     Konteks: {context_biasa[:800]}...

#     ---------------------------------------------------------------------------
#     [SISTEM B - HYBRID RAG]
#     Jawaban: {ans_hybrid}
#     Konteks: {context_hybrid[:800]}...

#     ---------------------------------------------------------------------------
#     HASIL EVALUASI (WAJIB FORMAT JSON):
#     {{
#         "Sistem_A": {{
#             "faithfulness": 0.0,
#             "answer_relevance": 0.0,
#             "context_precision": 0.0
#         }},
#         "Sistem_B": {{
#             "faithfulness": 0.0,
#             "answer_relevance": 0.0,
#             "context_precision": 0.0
#         }},
#         "catatan_analisis": "Penjelasan singkat mengapa salah satu lebih unggul."
#     }}
#     """

#     response = llm.invoke(prompt_eval)
#     try:
#         # Membersihkan output JSON dari backticks
#         clean_json = response.content.replace('```json', '').replace('```', '').strip()
#         return json.loads(clean_json)
#     except:
#         return None

# # =============================================================
# # DATASET UJI (DARI DAFTAR PERTANYAAN MANUAL ANDA)
# # =============================================================
# pertanyaan_anda = [
#     "Mengapa para filsuf menyatakan bahwa sesuatu yang berawal, mustahil lahir dari yang kekal secara mutlak?",
#     "Apa yang dimaksud dengan tajaddud, murad, dan iradah?",
#     "Bagaimana pendapat Al-Ghazali mengenai argumen Galen tentang penyusutan matahari?",
#     "Siapa saja ahli kalam yang berusaha memecahkan masalah kemustahilan?",
#     "Bagaimana pendapat para filsuf mengenai Allah adalah pencipta dan apa sanggahan Al-Ghazali?"
# ]

# rekap_hasil = []

# print("🧪 Menjalankan Evaluasi 3 Metrik RAGAS... Mohon tunggu.\n")

# for q in pertanyaan_anda:
#     print(f"🔎 Menguji: {q[:50]}...")

#     # 1. Jalankan RAG Biasa (Simpan Jawaban & Konteks)
#     # Catatan: Pastikan fungsi rag_chat Anda me-return jawaban string
#     res_biasa = rag_chat(q)

#     # 2. Jalankan Hybrid RAG (Simpan Jawaban & Konteks)
#     res_hybrid = hybrid_rag_chat(q)

#     # 3. Hitung Metrik dengan Judge AI
#     skor = evaluasi_ragas_judge(q, res_biasa, res_hybrid, "Data ChromaDB", "Data ChromaDB + Neo4j")

#     if skor:
#         rekap_hasil.append({
#             "Pertanyaan": q,
#             "A_Faithfulness": skor['Sistem_A']['faithfulness'],
#             "A_Relevance": skor['Sistem_A']['answer_relevance'],
#             "A_Precision": skor['Sistem_A']['context_precision'],
#             "B_Faithfulness": skor['Sistem_B']['faithfulness'],
#             "B_Relevance": skor['Sistem_B']['answer_relevance'],
#             "B_Precision": skor['Sistem_B']['context_precision'],
#             "Analisis": skor['catatan_analisis']
#         })

# # --- OUTPUT TABEL UNTUK BAB 4 ---
# df_final = pd.DataFrame(rekap_hasil)

# # Menampilkan Rata-rata Skor Keseluruhan
# print("\n" + "="*40)
# print("📊 RATA-RATA SKOR EVALUASI (0.0 - 1.0)")
# print("="*40)
# print(df_final[["A_Faithfulness", "B_Faithfulness", "A_Relevance", "B_Relevance", "A_Precision", "B_Precision"]].mean())

# # Tampilkan Tabel Detail
# display(df_final)

In [33]:
import shutil

# Nama folder yang mau di download
folder_path = '/content/chroma_db_tahafut'

# Nama file zip output
output_path = '/content/chroma_db_tahafut.zip'

# Zip folder
shutil.make_archive(output_path.replace('.zip', ''), 'zip', folder_path)
from google.colab import files

files.download('/content/chroma_db_tahafut.zip')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>